# Intervention Drift Analysis

This notebook loads preserved OQI selected-feature parity artifacts, summarizes retained-feature and tracked-logit drift, and inspects how the strongest layer-specific errors appear to propagate through retained feature rows.

Typical artifact generation flow:

```bash
export IT_REPO_DIR=${HOME}/repos/interpretune
export IT_VENV_BASE=/mnt/cache/${USER}/.venvs
export IT_TARGET_VENV=it_latest

cd "${IT_REPO_DIR}" && \
  IT_RUN_STANDALONE_TESTS=1 \
  IT_PARITY_PRESERVE_ARTIFACTS=1 \
  IT_PARITY_PRESERVE_ARTIFACT_DIR="/tmp/it_concept_direction_analysis_artifacts" \
  "${IT_VENV_BASE}/${IT_TARGET_VENV}/bin/python" -m pytest \
  tests/core/test_analysis_backend_parity.py \
  -k 'oqi_debug_selected_feature_adjacency_trace' -vv
```

In [ ]:
# ruff: noqa: E402
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "tests").exists():
            return candidate
    raise RuntimeError(f"Unable to locate repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.it_examples.utils.nb_ui_utils import display_layer_divergence_summary, display_logit_drift_summary
from src.it_examples.utils.raw_graph_analysis import plot_ridgeline_convergence
from tests.nb_experiments.concept_direction.analysis.intervention_drift_analysis import (
    PRESERVE_ARTIFACT_DIR_ENV,
    build_cli_summary,
    build_layer_error_distribution,
    load_preserved_intervention_artifacts,
)

In [ ]:
artifact_root = Path(
    os.environ.get(
        PRESERVE_ARTIFACT_DIR_ENV,
        REPO_ROOT / "tests" / "nb_experiments" / "concept_direction" / "analysis" / "artifacts",
    )
)
artifact_dirs = sorted([path for path in artifact_root.glob("*") if path.is_dir()])
if not artifact_dirs:
    raise FileNotFoundError(f"No preserved parity artifacts found under {artifact_root}")
artifact_dir = artifact_dirs[-1]
artifact_dir

In [ ]:
artifacts = load_preserved_intervention_artifacts(artifact_dir)
summary = build_cli_summary(artifacts, target_layer=33, top_k_rows=5, top_k_sources=5)
summary["feature_row"], summary["report"]["layer_with_max_divergence"]

In [ ]:
display_layer_divergence_summary(summary["report"]["layer_summaries"])
display_logit_drift_summary(summary["report"]["logit_summary"])
summary["report"]["top_feature_errors"][:10]

In [ ]:
layer_error_distribution = build_layer_error_distribution(artifacts.report)
plot_ridgeline_convergence(
    layer_error_distribution,
    title="Absolute retained-feature activation error distribution by layer",
)

In [ ]:
summary["propagation"]

In [ ]:
print(json.dumps(summary, indent=2))